# 71 · 생성 전 검색·근거 완전성 검사

새API0. 기존70결과와 원문은 보존한다. 검색 결과 자체와 LLM 입력에 구조화 근거를 붙이는 효과를 분리한다. 질문별 요구 계약을 먼저 고정하고 입력 의미 판정이 끝난 뒤에만 기존 답변을 진단용 연결한다. 사후 개발 점검이며 독립 확인시험이 아니다.

공유 집계 helper는 같은 카드 AND·N/A·인용 경계를 API 없이 회귀검사하기 위해 scripts에 둔다. HTML은 main이 직접 작성한다. 계획: notes/71_pre_generation_evidence_audit_plan.md.


In [1]:
from pathlib import Path
import json,hashlib,importlib.util,time,resource,copy
from collections import Counter,defaultdict
ROOT=Path.cwd() if (Path.cwd()/'notebooks').is_dir() else Path.cwd().parent
BASE=ROOT/'notebooks/data/71_pre_generation_evidence_audit'
PREV=ROOT/'notebooks/data/70_cap_context_repair'
BASE.mkdir(parents=True,exist_ok=True)
def read(path):return json.loads(Path(path).read_text())
def rawsha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write(name,value):
 path=BASE/name
 content=json.dumps(value,ensure_ascii=False,indent=2)+'\n'
 if path.exists() and path.read_text()!=content:raise RuntimeError('Refuse to overwrite frozen artifact: '+str(path))
 path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
def load_helper():
 spec=importlib.util.spec_from_file_location('evidence71',ROOT/'notebooks/scripts/71_evidence_metrics.py')
 m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m);return m
M=load_helper()
print('Offline71: no model, index, retrieval or answer API calls')


Offline71: no model, index, retrieval or answer API calls


In [2]:
qcontract=read(PREV/'question_contract.json')
source_union=read(ROOT/read(PREV/'evaluation_context.json')['source_union_path'])
assert len(source_union['records'])==717
DEFS={
'C01':('same_card',{'coffee':'커피 결제에 실제 할인 또는 명시적 캐시백이 적용되는 경로'}),
'C02':('same_card',{'netflix':'넷플릭스 결제에 실제 할인 또는 명시적 캐시백이 적용되는 경로'}),
'C03':('same_card',{'dental':'치과 치료비에 실제 할인 또는 명시적 캐시백이 적용되는 경로'}),
'C04':('same_card',{'cinema':'영화관 이용에 적용되는 할인 경로. 원문에 없는 주말 특별율은 요구하지 않는다'}),
'C05':('same_card',{'phone_autopay':'휴대폰 요금 자동납부에 적용되는 할인 경로. 결제와 자동납부 관계까지 확인'}),
'C06':('same_card',{'convenience':'편의점 결제에 적용되는 할인 경로'}),
'C07':('same_card',{'electricity':'전기요금에 적용되는 할인 경로. 전기차 충전은 대체 불가'}),
'C08':('same_card',{'delivery':'저녁 배달앱 결제에 적용되는 할인 경로. 시간제 혜택이면 해당 시간 포함'}),
'C09':('same_card',{'coffee':'커피 할인 경로','netflix':'같은 카드의 넷플릭스 할인 경로','compatible':'두 경로가 배타적 선택팩이 아닌 동시에 가능한 조합'}),
'C10':('same_card',{'convenience':'편의점 할인 경로','delivery':'같은 카드의 배달앱 할인 경로','compatible':'두 경로가 배타적 선택팩이 아닌 동시에 가능한 조합'}),
'C11':('same_card',{'issuer':'질문의 우리카드 발급사 범위','hospital':'병원비에 적용되는 할인 또는 명시적 캐시백 경로'}),
'C12':('same_card',{'coffee':'커피 할인 경로','regular_spend':'그 경로의 정규 최소 전월실적이 30만원 이하. 신규 유예만으로 충족시키지 않음'}),
'C13':('same_card',{'netflix':'넷플릭스 할인 경로','no_regular_spend':'해당 경로의 정규 전월실적 조건 없음이 원문과 입력에 명시됨. 미기재와 구별'}),
'C14':('same_card',{'veterinary':'동물병원 할인 또는 명시적 캐시백 경로','spend_time':'전월60만원으로 해당 경로의 최소실적 충족 또는 실제 무실적. 당월/최근연속실적을 전월로 대체하지 않음'}),
'C15':('same_card',{'netflix_target':'지정 SIMPLE 카드의 넷플릭스 혜택 관계','payment_channel':'같은 혜택의 대상 가맹점 공식 홈페이지/앱 직접 신청 결제 경로','rate':'같은 디지털 혜택 할인율 1.5%. 기본0.8%를 디지털율로 전이하지 않음'}),
'C16':('same_card',{'delivery_target':'I-ALL의 배달앱 LIVING 혜택 관계','convenience_target':'같은 카드의 편의점 LIVING 혜택 관계','spend':'해당 LIVING 일반 전월50만원/우대 최근3개월 매달50만원의 관계','delivery_channel':'배달앱 공식 앱/홈페이지 직접 결제 등 실제 배달 경로','convenience_channel':'편의점 오프라인 등 실제 적용 경로와 온라인(간편결제 등) 제외 문맥'}),
'C17':('same_card',{'starbucks':'그린카드v2의 스타벅스 커피 혜택 관계','regular_spend':'같은 커피 혜택 최소 전월실적20만원','monthly_frequency':'같은 커피 제공처 통합 월2회 관계'}),
'C18':('same_card',{'netflix_rate':'지정 Discount Plan+ 넷플릭스와20% 정기결제 할인율 관계','eligible_amount':'할인받기 전 대상 이용금액 건1만원 상한. 지급할인액1만원이 아님','monthly_cap':'전월60만원 해당 정기결제 월공유한도5천원 관계','qualification_channel':'전월60만원 해당 정규실적/공식 홈페이지 정기결제 자격 관계'}),
'C19':('same_card',{'living_path':'생활비/장보기 등 넓은 목적에 맞는 실제 한 카드의 사용 대상과 혜택 경로. 고정 대표카드/수치 강제 없음'}),
'C20':('comparison',{'travel_first':'첫 번째 카드의 여행 관련 실제 혜택 경로','travel_second':'다른 카드의 여행 관련 실제 혜택 경로','comparison':'두 경로의 대상/지급형태/조건 중 원문으로 확인되는 비교 가능한 차이. 미요청율/한도 전부는 요구하지 않음'}),
'C21':('same_card',{'online_shopping':'온라인 물품 구매에 적용되는 할인/적립 등 원문상 유효한 카드 경로. 임의 지정 플랫폼 강제 없음'}),
'C22':('not_applicable',{}),'C23':('not_applicable',{}),'C24':('not_applicable',{})}
records=[]
for q in qcontract['records']:
 kind,units=DEFS[q['query_id']]
 records.append(dict(query_id=q['query_id'],question=q['question'],question_type=q['question_type'],path_kind=kind,units=list(units),unit_definitions=units,inherited_direct_requirements=q['requirements'],na_reason='C22:일반선택기준/C23:개인남은횟수/C24:보유상품미지정은 카드검색점수로 답변태도를 대신하지 않음' if not units else None))
value=dict(status='frozen_before71_input_semantic_review',records=records,source_union_path=read(PREV/'evaluation_context.json')['source_union_path'],source_union_raw_sha256=rawsha(ROOT/read(PREV/'evaluation_context.json')['source_union_path']),question_contract70_raw_sha256=rawsha(PREV/'question_contract.json'),reference_scope='Same717globalsourcefacts_allqueries; acceptable paths are open, not exhaustive gold card membership',policies=['Unaskedconditionsnotmandatory','SamecardANDnotsyntheticunion','Actualserializedevidenceonly; owner/rate/cap/relationmustmatchsource','Unknownnotzero; ambiguousandcontradictednotfullsupport','C22-C24diagnosticNA;21factualquestionsprimary','NoLLManswersemanticreaduntilinputreviewssealed','Prior70outcomesalreadyknown;notblindedindependenttest'],new_api_calls=0)
write('question_contract.json',value)
write('contract_seal.json',dict(question_contract_raw_sha256=rawsha(BASE/'question_contract.json'),question_contract_sha256=M.sha(value),source_union_raw_sha256=value['source_union_raw_sha256'],status='before_input_review'))
print('Frozen24questions/21factual; units=',sum(len(q['units'])for q in records))


Frozen24questions/21factual; units= 42


In [3]:
contract=read(BASE/'question_contract.json')
seal=read(BASE/'contract_seal.json');assert seal['question_contract_raw_sha256']==rawsha(BASE/'question_contract.json')
context70=read(PREV/'evaluation_context.json');before={**context70['files_sha256'],str((PREV/'evaluation_context.json').relative_to(ROOT)):rawsha(PREV/'evaluation_context.json')}
for name in ('jobs.json','retrieval_results.json','execution_manifest.json','question_contract.json','evaluation_results.json','evaluation_summary.json','manual_reviews.json'):
 before[str((PREV/name).relative_to(ROOT))]=rawsha(PREV/name)
for path,digest in before.items():assert rawsha(ROOT/path)==digest,path
write('preservation_before.json',dict(files_sha256=before,operating_and_original_artifacts_read_only=True))
rows=read(PREV/'retrieval_results.json')['records'];jobs=read(PREV/'jobs.json')['records'];byjobs={r['identity']:r for r in jobs}
unique={};bindings=[];paired=[]
for row in rows:
 if row['mode']not in('SEQ65','SEQ_REPAIRED'):continue
 job=byjobs[row['identity']];prefix=row['result']['evidence'];after=job['result']['evidence']
 assert after[:len(prefix)]==prefix
 arms={'BASE':job['result']} if row['mode']=='SEQ65' else {'FILTER':row['result'],'FACT':job['result']}
 ids={}
 for arm,result in arms.items():
  canonical=dict(query_id=row['query_id'],question=row['question'],profile=row['profile'],cards=result['cards'],evidence=result['evidence'])
  uid=M.sha(canonical);packet=dict(input_id=uid,**canonical)
  if uid in unique:assert unique[uid]==packet
  unique[uid]=packet
  bindings.append(dict(identity=row['identity'],query_id=row['query_id'],profile=row['profile'],repeat=row['repeat'],arm=arm,input_id=uid,job_raw_sha256=rawsha(PREV/'jobs.json'),source_result_sha256=M.sha(result),prompt_binding='Both FILTER/FACT use actual modified job instructions; FILTER removes only structured suffix' if arm!='BASE' else 'actual original baseline job input'))
  ids[arm]=uid
 if row['mode']=='SEQ_REPAIRED':paired.append(dict(query_id=row['query_id'],profile=row['profile'],repeat=row['repeat'],FILTER=ids['FILTER'],FACT=ids['FACT'],added=len(after)-len(prefix),no_eviction=True))
assert len(bindings)==432 and len(paired)==144
write('inputs.json',dict(records=list(unique.values()),bindings=bindings,pairs=paired,synthetic_FILTER_not_called=True))
write('source_catalog.json',read(ROOT/contract['source_union_path']))
for q in contract['records']:
 entries=[p for p in unique.values()if p['query_id']==q['query_id']]
 write('packets/'+q['query_id']+'.json',dict(contract=q,inputs=entries,binding=[v for v in bindings if v['query_id']==q['query_id']]))
write('pre_review_seal.json',dict(contract_raw_sha256=rawsha(BASE/'question_contract.json'),inputs_raw_sha256=rawsha(BASE/'inputs.json'),source_catalog_raw_sha256=rawsha(BASE/'source_catalog.json'),unique_inputs=len(unique),logical_inputs=432,old_answers_read_for_semantic_review=False))
print('Prepared',len(unique),'unique inputs; 432logicalviews;144fixedA/Bpairs')


Prepared 89 unique inputs; 432logicalviews;144fixedA/Bpairs


In [4]:
start=time.perf_counter();cpu=time.process_time()
contract=read(BASE/'question_contract.json');definition={q['query_id']:q for q in contract['records']}
field_policy=read(BASE/'validation_contract.json')['unit_input_fields']
assert field_policy=={q['query_id']:{u:['text','issuer','card_name']if(q['query_id']=='C11'and u=='issuer')else['text']for u in q['units']}for q in contract['records']}
inputs=read(BASE/'inputs.json');packets={p['input_id']:p for p in inputs['records']}
catalog={r['condition_id']:r for r in read(BASE/'source_catalog.json')['records']}
assert len(catalog)==717 and Counter(r['source_status']for r in catalog.values())=={'verified':716,'uncertain':1}
# Preserve the old uncertain Pay boundary; it cannot certify positive support.
assert rawsha(BASE/'question_contract.json')==read(BASE/'contract_seal.json')['question_contract_raw_sha256']
assert rawsha(BASE/'inputs.json')==read(BASE/'pre_review_seal.json')['inputs_raw_sha256']
files=sorted((BASE/'manual_input_reviews').glob('C*.json'));assert len(files)==24
reviews={};origins={}
for file in files:
 bundle=read(file);qid=file.stem
 assert bundle['question_contract_sha256']==M.sha(definition[qid])
 for review in bundle['records']:
  key=review['input_id'];assert key not in reviews
  M.validate(review,definition[qid],packets[key],catalog,field_policy[qid])
  assert review['query_id']==qid==packets[key]['query_id']
  assert review['profile']==packets[key]['profile']
  reviews[key]=review;origins[key]=dict(path=str(file.relative_to(ROOT)),raw_sha256=rawsha(file),record_sha256=M.sha(review))
assert set(reviews)==set(packets) and len(reviews)==89
for pair in inputs['pairs']:
 a,b=packets[pair['FILTER']],packets[pair['FACT']]
 assert a['question']==b['question'] and a['cards']==b['cards']
 assert b['evidence'][:len(a['evidence'])]==a['evidence'] and len(b['evidence'])-len(a['evidence'])==pair['added']
 def eligibility(key):
  return {M.path_key(v):v['eligibility'] for v in reviews[key]['paths']}
 assert eligibility(pair['FILTER'])==eligibility(pair['FACT']),pair
for file,digest in read(BASE/'preservation_before.json')['files_sha256'].items():assert rawsha(ROOT/file)==digest,file
write('input_reviews.json',dict(status='manually_reviewed_sources_and_actual_inputs_before71_answer_linkage',records=list(reviews.values()),origins=origins,semantics_by_reviewers_not_hash_inferred=True))
seal=dict(status='input_judgments_sealed_before_answer_linkage',files_sha256={str(file.relative_to(ROOT)):rawsha(file)for file in files},question_contract_raw_sha256=rawsha(BASE/'question_contract.json'),inputs_raw_sha256=rawsha(BASE/'inputs.json'),reviews_raw_sha256=rawsha(BASE/'input_reviews.json'),source_catalog_raw_sha256=rawsha(BASE/'source_catalog.json'),helper_raw_sha256=rawsha(ROOT/'notebooks/scripts/71_evidence_metrics.py'),validation_contract_raw_sha256=rawsha(BASE/'validation_contract.json'),exposure_disclosure_raw_sha256=rawsha(BASE/'review_exposure_disclosure.json'),unique_reviews=89,logical_views=432,prior_result_knowledge_disclosed=True)
write('input_review_seal.json',seal)
print('Verified89unique input reviews/432views; contract/source/quotes/owners/paired scope/original files verified; sealed before output linkage')


Verified89unique input reviews/432views; contract/source/quotes/owners/paired scope/original files verified; sealed before output linkage


In [5]:
rows=[]
for bind in inputs['bindings']:
 review=reviews[bind['input_id']];q=definition[bind['query_id']]
 rows.append(dict(**bind,**M.score(review,q),review_sha256=M.sha(review),question_type=q['question_type'],required_units=len(q['units']),path_status_counts=dict(Counter(status for v in review['paths']for status in v['units'].values()))))
assert len(rows)==432 and len({(r['query_id'],r['profile'],r['repeat'],r['arm'])for r in rows})==432
conditions=[]
for profile in sorted({r['profile']for r in rows}):
 for arm in ('BASE','FILTER','FACT'):
  rs=[r for r in rows if r['profile']==profile and r['arm']==arm]
  factual=[r for r in rs if r['required_units']];known=[r for r in factual if r['coverage']is not None]
  conditions.append(dict(profile=profile,arm=arm,logical_inputs=len(rs),factual_inputs=len(factual),na_inputs=len(rs)-len(factual),unknown_inputs=len(factual)-len(known),known_card_hit=sum(r['card_hit']is True for r in factual),card_hit_known_denominator=sum(r['card_hit']is not None for r in factual),complete=sum(r['complete']is True for r in factual),complete_known_denominator=len(known),coverage_macro=sum(r['coverage']for r in known)/len(known)if known else None,coverage_definition='Mean best verified path fraction across factual question/repeat; not pooled tokens/chunks/all relevant card recall',unique_inputs=len({r['input_id']for r in rs}),unresolved_candidate_path_inputs=sum(r['unresolved_paths']>0 for r in factual)))
write('evidence_results.json',dict(records=rows,input_review_seal_sha256=M.sha(read(BASE/'input_review_seal.json'))))
write('evidence_summary.json',dict(conditions=conditions,primary_factual_questions=21,all_questions=24,na_queries=['C22','C23','C24'],unknown_not_zero=True,partial_gold_not_exhaustive_card_recall=True,new_api_calls=0))
print(json.dumps(conditions,ensure_ascii=False,indent=2))


[
  {
    "profile": "card_page_section_benefit",
    "arm": "BASE",
    "logical_inputs": 72,
    "factual_inputs": 63,
    "na_inputs": 9,
    "unknown_inputs": 0,
    "known_card_hit": 63,
    "card_hit_known_denominator": 63,
    "complete": 63,
    "complete_known_denominator": 63,
    "coverage_macro": 1.0,
    "coverage_definition": "Mean best verified path fraction across factual question/repeat; not pooled tokens/chunks/all relevant card recall",
    "unique_inputs": 26,
    "unresolved_candidate_path_inputs": 6
  },
  {
    "profile": "card_page_section_benefit",
    "arm": "FILTER",
    "logical_inputs": 72,
    "factual_inputs": 63,
    "na_inputs": 9,
    "unknown_inputs": 0,
    "known_card_hit": 63,
    "card_hit_known_denominator": 63,
    "complete": 63,
    "complete_known_denominator": 63,
    "coverage_macro": 1.0,
    "coverage_definition": "Mean best verified path fraction across factual question/repeat; not pooled tokens/chunks/all relevant card recall",
    "uni

In [6]:
by={(r['query_id'],r['profile'],r['repeat'],r['arm']):r for r in rows}
comparisons=[];unit_transitions=[]
for profile in sorted({r['profile']for r in rows}):
 for label,a,b in [('actual_pipeline','BASE','FACT'),('retrieval_without_facts','BASE','FILTER'),('facts_only','FILTER','FACT')]:
  qs=[]
  for q in definition:
   pairs=[(by[q,profile,t,a],by[q,profile,t,b])for t in(1,2,3)]
   if not definition[q]['units']:
    qs.append(dict(query_id=q,na=True));continue
   valid=[(x,y)for x,y in pairs if x['coverage']is not None and y['coverage']is not None]
   delta=sum(y['coverage']-x['coverage']for x,y in valid)/len(valid)if valid else None
   qs.append(dict(query_id=q,na=False,paired_known=len(valid),baseline_complete=sum(x['complete']is True for x,y in valid),candidate_complete=sum(y['complete']is True for x,y in valid),delta_coverage=delta,comparison='win'if delta is not None and delta>1e-12 else'loss'if delta is not None and delta< -1e-12 else'tie'if delta is not None else'unknown'))
  comparisons.append(dict(profile=profile,comparison=label,baseline=a,candidate=b,question_wlt=dict(Counter(q['comparison']for q in qs if not q['na'])),questions=qs))
for pair in inputs['pairs']:
 if not definition[pair['query_id']]['units']:continue
 ar={M.path_key(v):v for v in reviews[pair['FILTER']]['paths']};br={M.path_key(v):v for v in reviews[pair['FACT']]['paths']}
 assert set(ar)==set(br)
 for key in ar:
  for unit in ar[key]['units']:
   before,after=ar[key]['units'][unit],br[key]['units'][unit]
   if before!=after:
    unit_transitions.append(dict(query_id=pair['query_id'],profile=pair['profile'],repeat=pair['repeat'],cards=list(key[0]),path_id=key[1],unit=unit,before=before,after=after,source_eligibility=ar[key]['eligibility'],FILTER=pair['FILTER'],FACT=pair['FACT'],new_quotes=br[key]['unit_evidence'][unit]['input_quotes'],reason=br[key]['unit_evidence'][unit]['reason']))
applied=[]
for profile in sorted({r['profile']for r in rows}):
 ps=[p for p in inputs['pairs']if p['profile']==profile and p['added']>0]
 for arm in('FILTER','FACT'):
  rs=[by[p['query_id'],profile,p['repeat'],arm]for p in ps];fs=[r for r in rs if r['required_units']];known=[r for r in fs if r['coverage']is not None]
  applied.append(dict(profile=profile,arm=arm,all_applied_pairs=len(ps),factual_pairs=len(fs),na_pairs=len(rs)-len(fs),complete=sum(r['complete']is True for r in fs),known_denominator=len(known),coverage_macro=sum(r['coverage']for r in known)/len(known)if known else None))
groups=[]
for profile in sorted({r['profile']for r in rows}):
 for arm in('BASE','FILTER','FACT'):
  for kind in sorted({q['question_type']for q in definition.values()}):
   rs=[r for r in rows if r['profile']==profile and r['arm']==arm and r['question_type']==kind];known=[r for r in rs if r['coverage']is not None]
   groups.append(dict(profile=profile,arm=arm,question_type=kind,logical_inputs=len(rs),known_denominator=len(known),complete=sum(r['complete']is True for r in known),coverage_macro=sum(r['coverage']for r in known)/len(known)if known else None))
write('comparison_analysis.json',dict(comparisons=comparisons,applied_only=applied,groups=groups,unit_transitions=unit_transitions,unit_transitions_not_independent_questions=True,query_repeats_not_independent=True,source_conflict70_diagnostics_preserved=True))
print('Compared3contrasts, questions/subgroups/applied subsets; changed card/unit records=',len(unit_transitions))


Compared3contrasts, questions/subgroups/applied subsets; changed card/unit records= 15


In [7]:
# This is the first intentional semantic read of old answer evaluations in71.
# The retrospective C14 technical-print exposure is separately disclosed.
seal=read(BASE/'input_review_seal.json')
assert seal['reviews_raw_sha256']==rawsha(BASE/'input_reviews.json')
for file,digest in seal['files_sha256'].items():assert rawsha(ROOT/file)==digest
old={r['identity']:r for r in read(PREV/'evaluation_results.json')['records']}
linked=[]
for r in rows:
 if r['arm']=='FILTER':continue
 answer=old[r['identity']]
 linked.append(dict(query_id=r['query_id'],profile=r['profile'],repeat=r['repeat'],arm=r['arm'],input_id=r['input_id'],input_complete=r['complete'],input_coverage=r['coverage'],required_units=r['required_units'],answer_technical=answer['technical_success'],answer_direct=answer['direct_request_satisfied'],answer_source_accuracy=answer['visible_factual_accuracy'],answer_pass=answer['passed'],visible_errors=answer['visible_errors'],old_evaluation_row_sha256=M.sha(answer)))
summary=[]
for profile in sorted({r['profile']for r in linked}):
 for arm in('BASE','FACT'):
  rs=[r for r in linked if r['profile']==profile and r['arm']==arm and r['required_units']]
  complete=[r for r in rs if r['input_complete']is True]
  summary.append(dict(profile=profile,arm=arm,factual_answer_pairs=len(rs),input_complete=len(complete),direct_satisfied_given_complete=sum(r['answer_direct']for r in complete),visible_accurate_given_complete=sum(r['answer_source_accuracy']for r in complete),answer_pass_given_complete=sum(r['answer_pass']for r in complete),complete_but_not_pass=sum(not r['answer_pass']for r in complete),incomplete_but_answer_pass=sum(r['answer_pass']for r in rs if r['input_complete']is False),not_unit_level_answer_reflection_rate=True))
write('generation_linkage.json',dict(summary=summary,records=linked,old_answer_evaluation_raw_sha256=rawsha(PREV/'evaluation_results.json'),input_judgment_sealed_before_linkage=True,not_a_new_generation_experiment=True,not_causal_fact_addition_generation_effect=True,clarification='Existing direct-request/visible-factual grades are linked diagnostics, not newly judged atomic answer reflection. FILTER has no model response.'))
for file,digest in read(BASE/'preservation_before.json')['files_sha256'].items():assert rawsha(ROOT/file)==digest,file
resources=dict(status='offline_final_evaluation_segment_only',wall_seconds=time.perf_counter()-start,cpu_seconds=time.process_time()-cpu,process_peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,new_api_requests=0,new_api_cost_usd='0',new_embedding_tokens=0,new_search_runs=0,new_model_answers=0,manual_review_time_seconds=None,online_latency_seconds=None,unmeasured=['manual review elapsed time','full notebook wall time','online response latency','GPU peak','invoice'],historical70_cost_and_latency_not_remeasured=True)
name='resource_runs/'+str(time.time_ns())+'.json';write(name,resources)
print(json.dumps(summary,ensure_ascii=False,indent=2));print('Resource receipt:',name,json.dumps(resources,ensure_ascii=False))


[
  {
    "profile": "card_page_section_benefit",
    "arm": "BASE",
    "factual_answer_pairs": 63,
    "input_complete": 63,
    "direct_satisfied_given_complete": 63,
    "visible_accurate_given_complete": 62,
    "answer_pass_given_complete": 62,
    "complete_but_not_pass": 1,
    "incomplete_but_answer_pass": 0,
    "not_unit_level_answer_reflection_rate": true
  },
  {
    "profile": "card_page_section_benefit",
    "arm": "FACT",
    "factual_answer_pairs": 63,
    "input_complete": 63,
    "direct_satisfied_given_complete": 62,
    "visible_accurate_given_complete": 57,
    "answer_pass_given_complete": 57,
    "complete_but_not_pass": 6,
    "incomplete_but_answer_pass": 0,
    "not_unit_level_answer_reflection_rate": true
  },
  {
    "profile": "parent_child_bundle",
    "arm": "BASE",
    "factual_answer_pairs": 63,
    "input_complete": 63,
    "direct_satisfied_given_complete": 59,
    "visible_accurate_given_complete": 61,
    "answer_pass_given_complete": 58,
    "comp